# 🎬 ReviewSense AI — Movie Review Sentiment Analysis Using LSTM

Welcome to the **ReviewSense AI** model training notebook! This notebook is designed to train a Long Short-Term Memory (LSTM) recurrent neural network on the classic IMDB Movie Reviews dataset to classify sentiments as **Positive** or **Negative**.

### 📋 Project Workflow
1. **Environment Setup & Random Seed Initialization**
2. **Dataset Loading (Keras IMDB 50,000 Reviews)**
3. **Data Preprocessing & Sequence Padding**
4. **LSTM Architecture Definition (Embedding -> SpatialDropout -> LSTM -> Dense -> Sigmoid)**
5. **Model Training & Early Stopping**
6. **Evaluation on Unseen Test Dataset & Plotting Metrics**
7. **Inference Verification on Custom Reviews**
8. **Exporting Artifacts for the Streamlit Application**

## 1. Setup & Imports
Ensure all necessary deep learning and data processing libraries are loaded, and fix random seeds for reproducible results.

In [ ]:
import os
import re
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Set reproducible seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")
gpu_devices = tf.config.list_physical_devices('GPU')
print(f"GPU Available: {len(gpu_devices) > 0} ({gpu_devices})")

## 2. Load the IMDB Movie Reviews Dataset
We load the top 10,000 most frequently occurring words in the IMDB dataset. The dataset contains 25,000 training reviews and 25,000 test reviews.

In [ ]:
VOCAB_SIZE = 10000    # Consider top 10,000 words
MAX_LEN = 200         # Pad/truncate reviews to 200 words
EMBEDDING_DIM = 128   # Embedding vector dimension
LSTM_UNITS = 64       # LSTM memory units
BATCH_SIZE = 64
EPOCHS = 5

print("Loading IMDB dataset...")
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)
print(f"Training samples: {len(x_train)}")
print(f"Testing samples : {len(x_test)}")

## 3. Data Preprocessing & Sequence Inspection
In Keras IMDB dataset, reviews are pre-encoded into integers where:
- `0`: `<PAD>`
- `1`: `<START>`
- `2`: `<UNK>` (Out-of-vocabulary word)
- `3`: Unused offset

Let's decode a sample review back to English and pad sequences.

In [ ]:
# Download vocabulary index
raw_word_index = tf.keras.datasets.imdb.get_word_index()
# Invert dictionary: id -> word
inverted_word_index = {value + 3: key for key, value in raw_word_index.items()}
inverted_word_index[0] = "<PAD>"
inverted_word_index[1] = "<START>"
inverted_word_index[2] = "<UNK>"
inverted_word_index[3] = "<UNUSED>"

sample_decoded = " ".join([inverted_word_index.get(i, "?") for i in x_train[0][:50]])
print("Sample Review (first 50 tokens):\n", sample_decoded)
print("Label:", "Positive (1)" if y_train[0] == 1 else "Negative (0)")

# Apply pre-padding
x_train_padded = pad_sequences(x_train, maxlen=MAX_LEN, padding="pre", truncating="pre")
x_test_padded = pad_sequences(x_test, maxlen=MAX_LEN, padding="pre", truncating="pre")
print(f"\nShape after padding: {x_train_padded.shape}")

## 4. Build LSTM Neural Network
The model architecture consists of:
1. **Embedding Layer**: Converts discrete word IDs into continuous 128-dimensional dense vectors.
2. **SpatialDropout1D**: Regularizes entire feature maps to prevent overfitting on specific words.
3. **LSTM Layer**: 64 recurrent units capable of maintaining sequential memory over long text spans.
4. **Dense Layer**: 32 units with ReLU activation for non-linear representation.
5. **Dropout**: Regularization to enhance generalization.
6. **Dense Output**: 1 unit with Sigmoid activation producing a probability score between 0 (Negative) and 1 (Positive).

In [ ]:
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    SpatialDropout1D(0.2),
    LSTM(LSTM_UNITS, dropout=0.2, recurrent_dropout=0.0),
    Dense(32, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 5. Model Training
We train the model using an 80/20 train-validation split with Early Stopping to prevent overfitting.

In [ ]:
early_stopping = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True, verbose=1)

history = model.fit(
    x_train_padded,
    y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=1
)

## 6. Evaluation & Metric Visualizations
Evaluate performance on the genuine 25,000 unseen test set reviews and plot loss/accuracy curves.

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test_padded, y_test, verbose=1)
print(f"\nTest Loss    : {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

# Plot Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs_range = range(1, len(history.history['loss']) + 1)

# Accuracy
ax1.plot(epochs_range, history.history['accuracy'], marker='o', label='Train Accuracy')
ax1.plot(epochs_range, history.history['val_accuracy'], marker='s', label='Validation Accuracy')
ax1.set_title('Accuracy vs Epochs')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.6)

# Loss
ax2.plot(epochs_range, history.history['loss'], marker='o', color='coral', label='Train Loss')
ax2.plot(epochs_range, history.history['val_loss'], marker='s', color='crimson', label='Validation Loss')
ax2.set_title('Loss vs Epochs')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('training_metrics.png', dpi=300)
plt.show()

## 7. Inference on Custom Movie Reviews
Test the trained model on novel, custom movie reviews.

In [ ]:
def predict_custom_review(text):
    # 1. Clean
    cleaned = re.sub(r'<[^>]+>', ' ', text)
    cleaned = re.sub(r'[^a-zA-Z\s]', ' ', cleaned).lower().strip()
    cleaned = re.sub(r'\s+', ' ', cleaned)
    
    # 2. Tokenize & Encode
    tokens = [1]  # START_INDEX
    for word in cleaned.split():
        rank = raw_word_index.get(word)
        if rank is not None and (rank + 3) < VOCAB_SIZE:
            tokens.append(rank + 3)
        else:
            tokens.append(2)  # UNK_INDEX
            
    # 3. Pad
    if len(tokens) >= MAX_LEN:
        padded = tokens[-MAX_LEN:]
    else:
        padded = [0] * (MAX_LEN - len(tokens)) + tokens
        
    # 4. Predict
    score = float(model.predict(np.array([padded]), verbose=0)[0][0])
    sentiment = 'Positive' if score >= 0.5 else 'Negative'
    conf = score if sentiment == 'Positive' else (1.0 - score)
    
    print(f"Review     : \"{text}\"")
    print(f"Sentiment  : {sentiment}")
    print(f"Confidence : {conf * 100:.2f}% (Score: {score:.4f})\n")

predict_custom_review("This movie was an absolute triumph! Brilliant acting and breathtaking visuals.")
predict_custom_review("Boring, terrible acting, uninspired script, and completely wasted my evening.")

## 8. Export Artifacts for Streamlit Deployment
Save the trained model and vocabulary mapping so they can be loaded by `app.py`.

In [ ]:
# Save model
model.save('imdb_lstm_model.keras')
print("Saved model to imdb_lstm_model.keras")

# Save vocabulary index
with open('word_index.json', 'w', encoding='utf-8') as f:
    json.dump(raw_word_index, f)
print("Saved vocabulary mapping to word_index.json")

# Download in Google Colab
try:
    from google.colab import files
    print("Triggering browser download in Google Colab...")
    files.download('imdb_lstm_model.keras')
    files.download('word_index.json')
except ImportError:
    print("Running locally: files are stored in the current working directory.")